In [ ]:
import numpy as np
import napari

from morphotrack import losses, networks, analysis, utils
from morphotrack.losses import approx_const_curvature_loss, approx_zero_torsion_loss, dirichlet_energy
from morphotrack.meshes import remesh_uniform, sample_points_between_surfaces

import torch
import matplotlib.pyplot as plt

import trimesh
from tqdm import tqdm

In [ ]:
# Load coordinates and vectors
npz = np.load('../data/tracks_dense.npz')
coords = npz['positions']
local_vectors = npz['vectors']
local_vectors = local_vectors / np.linalg.norm(local_vectors, axis=1, keepdims=True)

fix_mesh = trimesh.load("../data/mesh_pia.ply")
mov_mesh = trimesh.load("../data/mesh_wm.ply")
fix_vertices = fix_mesh.vertices
mov_vertices = mov_mesh.vertices
fix_face = fix_mesh.faces
mov_face = mov_mesh.faces

### Or remesh if necessary
# target_area = 5000
# mov_mesh = remesh_uniform(mov_mesh, target_area)
# print(f"number of vertices: {mov_mesh.vertices.shape[0]}")
# mov_vertices = mov_mesh.vertices
# mov_face = mov_mesh.faces

mov_test = np.asarray(mov_vertices)

### Parameter settings
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Predict unit velocity field

In [ ]:
lr=1e-3

mixed_orientation = False
flipvector = False

v_field = networks.SimpleMLP(hidden_sizes=[256, 128, 64], use_norm=True,
                              use_residual=True, activation_func='SiLU').to(device)

optimizer = torch.optim.AdamW(v_field.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5000, min_lr=1e-5)
loss_log = {'align': [], 'divergence': [], 'curl': [], 'laplacian': [], 'dirichlet': [], 'curvature': [], 'torsion': []}

vec_pos = torch.from_numpy(coords).to(device, dtype=torch.float32)

### bbox bounds for normalization
all_verts_np = np.vstack([mov_mesh.vertices, fix_mesh.vertices])
pos_min = torch.from_numpy(all_verts_np.min(axis=0)).to(device, dtype=torch.float32)
pos_max = torch.from_numpy(all_verts_np.max(axis=0)).to(device, dtype=torch.float32)

vec_pos_norm = (vec_pos - pos_min) / (pos_max - pos_min)  # [0, 1]

dGdt_unit = torch.from_numpy(local_vectors).to(device, dtype=torch.float32)

fix_original = torch.from_numpy(np.asarray(fix_vertices)).to(device, dtype=torch.float32)
mov_test_original = torch.from_numpy(np.asarray(mov_test)).to(device, dtype=torch.float32)

if flipvector:
    local_vectors = -local_vectors

checkpoint_path = '../data/vhat_field_checkpoint.pth'
best_loss = float('inf')  # Initialize best loss

# Generate the random points between the two surfaces
r_points_original_np = sample_points_between_surfaces(fix_mesh, mov_mesh, n_candidates=500000, use_bbox=True)  
r_points_original = torch.from_numpy(r_points_original_np).to(device, dtype=torch.float32)
r_points_norm = (r_points_original - pos_min) / (pos_max - pos_min)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_vectors(np.stack([coords, local_vectors], axis=1), length=300., edge_width=100, edge_color='red', name='3D Vectors',out_of_slice_display=True)
viewer.add_points(r_points_original_np,size=30,face_color='white',blending='additive',out_of_slice_display=True)
viewer.add_surface((mov_vertices, mov_face), name='WM Mesh', blending='translucent',shading='smooth', colormap="magenta")
viewer.add_surface((fix_vertices, fix_face), name='Pia Mesh', blending='additive',shading='smooth', colormap="green")

In [ ]:
## Parameter settings for training
n_iters=500000
lambda_align = 1.0
lambda_divergence = 0
lambda_curl = 0
lambda_dirichlet = 1e-2 # Make it look Laplacian harmonic
lambda_curvature = 0
lambda_torsion = 1e-6 # Tortion free regularization
sampling_size = 5000

In [ ]:
# -----------------------
# Train
# -----------------------

v_field.train()
for it in range(n_iters):
    optimizer.zero_grad()

    # Compute alignment loss
    v_pred = v_field(vec_pos_norm)
    v_pred_norm = v_pred / (v_pred.norm(dim=1, keepdim=True))
    cos_sim = torch.sum(v_pred_norm * dGdt_unit, dim=1)
    loss_align = (1 - cos_sim).mean()

    # Get position of the trajectory for smoothness loss
    idx = torch.randint(len(r_points_norm), (sampling_size,))
    r_points = r_points_norm[idx]
    r_points = r_points.clone().requires_grad_(True)

    # For Laplacian
    velocities = v_field(r_points)
    mag_v   = velocities.norm(dim=1).clamp(min=1e-6)
    N, D = velocities.shape

    jacobian = []

    for i in range(D):  # vx, vy, vz
        grad_i = torch.autograd.grad(
            outputs=velocities[:, i],
            inputs=r_points,
            grad_outputs=torch.ones_like(velocities[:, i]),
            create_graph=True,
            retain_graph=True,
            only_inputs=True,
        )[0]  # Shape: [N, 3]
        jacobian.append(grad_i)

    # Stack into Jacobian tensor [N, 3, 3]
    jacobian = torch.stack(jacobian, dim=1)

    # Divergence Loss
    divergence = torch.stack([
        jacobian[:, 0, 0],  # dvx/dx
        jacobian[:, 1, 1],  # dvy/dy
        jacobian[:, 2, 2],  # dvz/dz
    ], dim=1).sum(dim=1)
    loss_divergence = ((divergence / mag_v) ** 2).mean() # (divergence ** 2).mean()

    # Curl Loss
    curl_x = jacobian[:, 2, 1] - jacobian[:, 1, 2]
    curl_y = jacobian[:, 0, 2] - jacobian[:, 2, 0]
    curl_z = jacobian[:, 1, 0] - jacobian[:, 0, 1]
    curl = torch.stack([curl_x, curl_y, curl_z], dim=1)
    loss_curl = ((curl ** 2).sum(dim=1) / mag_v.pow(2)).mean() # (curl ** 2).sum(dim=1).mean()

    loss_dirichlet = dirichlet_energy(v_field, r_points)
    loss_curvature = approx_const_curvature_loss(v_field, r_points, epsilon=1e-3)
    loss_torsion = approx_zero_torsion_loss(v_field, r_points, epsilon=1e-3)


    # Compute total loss
    loss = (lambda_align * loss_align
            + lambda_divergence * loss_divergence
            + lambda_curl * loss_curl
            + lambda_dirichlet * loss_dirichlet
            + lambda_curvature * loss_curvature
            + lambda_torsion * loss_torsion
            )


    loss.backward()
    torch.nn.utils.clip_grad_norm_(v_field.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step(loss_align.item())

    # Logging
    loss_log['align'].append(loss_align.item())
    loss_log['divergence'].append(loss_divergence.item())
    loss_log['curl'].append(loss_curl.item())
    loss_log['dirichlet'].append(loss_dirichlet.item())
    loss_log['curvature'].append(loss_curvature.item())
    loss_log['torsion'].append(loss_torsion.item())


    if it % 50 == 0:
        msg = (f"[{it}] Total: {loss.item():.4f}, "
               f"Align: {loss_align.item():.4f}, "
               f"Div: {loss_divergence.item():.4f}, "
               f"Curl: {loss_curl.item():.4f}, "
               f"Dirichlet: {loss_dirichlet.item():.4e}, "
               f"Curvature: {loss_curvature.item():.4e}, "
               f"Torsion: {loss_torsion.item():.4e}, "
               )

        print(msg)


    # Checkpoint saving
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save({
            'iteration': it,
            'model_state_dict': v_field.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'pos_min': pos_min.cpu(),
            'pos_max': pos_max.cpu(),
        }, checkpoint_path)

In [ ]:
iters = range(len(loss_log['align']))

fig, axes = plt.subplots(ncols=3, figsize=(14, 5))  # 2 plots side-by-side

# Plot Match Loss on the left
axes[0].plot(iters, loss_log['align'], label='Align Loss')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')
axes[0].set_title('Loss Over Iterations')
axes[0].legend()
axes[0].grid(True)


axes[1].plot(iters, loss_log['divergence'], label='Divergence Loss')
axes[1].plot(iters, loss_log['curl'], label='Curl Loss')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')
axes[1].set_title('Loss Over Iterations')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(iters, loss_log['curvature'], label='Curvature Loss')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_yscale('log')
axes[2].set_title('Loss Over Iterations')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Load v_field checkpoint
checkpoint_path = '../data/vhat_field_checkpoint.pth'
v_field_ckpt = torch.load(checkpoint_path, map_location='cuda')
v_field.load_state_dict(v_field_ckpt['model_state_dict'])
v_field.eval()  # Set to evaluation mode

In [ ]:
# -----------------------
# Visualization
# -----------------------
v_field_norm = networks.NormalizedField(v_field, pos_min, pos_max)

v_field_norm.eval()
thickness = 4000 # in microns
steps = 50
dt = thickness/steps
max_lines= 500

with torch.no_grad():
    mov_traj = analysis.integrate_rk4(mov_test_original, v_field_norm, steps=steps, dt=dt)

    X_all = mov_traj.detach().cpu().numpy()  # (T+1, N, 3)
    _, N, _ = X_all.shape

    # Determine which trajectories to draw as lines
    idx = utils.random_idx_with_max(N, max_lines)
    edge_width = 4

    viewer = napari.Viewer(ndisplay=3)
    
    shapes = [X_all[:, i, :] for i in range(X_all.shape[1])] if idx is None else [X_all[:, i, :] for i in idx]

    # Sampled initial points (magenta)
    viewer.add_shapes(shapes, shape_type='path', edge_color='yellow', edge_width=edge_width, name='Trajectories', visible=False)
    viewer.add_surface((mov_vertices, mov_face), name='WM Mesh', colormap="magenta", blending='translucent',shading='smooth')
    viewer.add_points(X_all[-1], name='X (t=1) (all)', size=50, face_color='red', blending='additive', visible=False)
    viewer.add_surface((fix_vertices, fix_face), name='Pia Mesh', colormap="green", blending='additive',shading='smooth')

    # Compute alignment loss and visualize orientation arrows
    v_pred = v_field_norm(vec_pos)
    alignment_sim = losses.alignment_loss(v_pred, dGdt_unit, norm_vector=True, norm_ref=True, mixed_orientation=mixed_orientation, return_vector=True).detach().cpu().numpy()
    cmap = plt.get_cmap('cool')  # You can use 'plasma', 'magma', etc.

    # Map scalar values to RGBA colors using the colormap
    alignment_sim_norm = alignment_sim / np.percentile(alignment_sim, 50)
    alignment_sim_norm[alignment_sim_norm>1] = 1
    colors = cmap(alignment_sim_norm)  # Output shape: (10, 4), with RGBA in [0, 1]
    viewer.add_vectors(np.stack([coords, local_vectors], axis=1), edge_color=colors, name='orientation_arrows', length=250, edge_width=25, visible=False)

    shapes = []
    for i in tqdm(range(mov_traj.shape[1])):
        trajectory = mov_traj[:,i,:].detach().cpu().numpy()
        hit, intersection_time = analysis.calculate_intersection(trajectory, fix_mesh)
        if intersection_time is not np.nan:
            if int(intersection_time) > 1:
                shapes.append(trajectory[:int(intersection_time)])

    idx = utils.random_idx_with_max(len(shapes), max_lines)
    shapes = [shapes[i] for i in idx]

    viewer.add_shapes(shapes, shape_type='path', edge_color='yellow', edge_width=edge_width, name='Trajectories')